In [ ]:
# --- Step 1: Setup ---
!pip install torch numpy pandas matplotlib scikit-learn statsmodels prophet --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.stattools import acf
from prophet import Prophet

# --- Step 2: Generate Large Synthetic Dataset ---
np.random.seed(42)
timesteps = 10000
t = np.arange(timesteps)

data = np.stack([
    0.05 * t + 10 * np.sin(0.1 * t) + np.random.normal(0, 1, timesteps),
    0.03 * t + 5 * np.cos(0.1 * t) + np.random.normal(0, 1, timesteps),
    0.02 * t + 8 * np.sin(0.2 * t) + np.random.normal(0, 1, timesteps)
], axis=1)

df = pd.DataFrame(data, columns=["feature1", "feature2", "feature3"])
df["timestamp"] = pd.date_range(start="2020-01-01", periods=timesteps, freq="H")

# --- Step 3: Preprocessing ---
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[["feature1", "feature2", "feature3"]])

def create_sequences(data, seq_len):
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        x = data[i:i+seq_len]
        y = data[i+seq_len]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_len = 48
X, y = create_sequences(scaled_data, seq_len)

X_train = torch.tensor(X[:9000], dtype=torch.float32)
y_train = torch.tensor(y[:9000], dtype=torch.float32)
X_test = torch.tensor(X[9000:], dtype=torch.float32)
y_test = torch.tensor(y[9000:], dtype=torch.float32)

# --- Step 4: Transformer Model ---
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, input_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = x[-1]
        return self.fc(x)

model = TimeSeriesTransformer(input_size=3)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- Step 5: Training ---
for epoch in range(50):
    model.train()
    output = model(X_train.transpose(0, 1))
    loss = criterion(output, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# --- Step 6: Evaluation Metrics ---
def mase(y_true, y_pred, y_naive):
    return np.mean(np.abs(y_true - y_pred)) / np.mean(np.abs(y_true - y_naive))

def theils_u(y_true, y_pred):
    num = np.sqrt(np.mean((y_true - y_pred)**2))
    denom = np.sqrt(np.mean(y_true**2)) + np.sqrt(np.mean(y_pred**2))
    return num / denom

model.eval()
with torch.no_grad():
    preds = model(X_test.transpose(0, 1)).numpy()
    true = y_test.numpy()
    naive = X_test[-1].numpy()

rmse = np.sqrt(mean_squared_error(true[:, 0], preds[:, 0]))
mase_score = mase(true[:, 0], preds[:, 0], naive[:, 0])
theil_u_score = theils_u(true[:, 0], preds[:, 0])

print(f"\nRMSE: {rmse:.4f}")
print(f"MASE: {mase_score:.4f}")
print(f"Theil’s U: {theil_u_score:.4f}")

# --- Step 7: Prophet Benchmark ---
prophet_df = pd.DataFrame({
    "ds": df["timestamp"],
    "y": df["feature1"]
})

prophet_model = Prophet()
prophet_model.fit(prophet_df[:9000])
future = prophet_model.make_future_dataframe(periods=1000, freq="H")
forecast = prophet_model.predict(future)

plt.plot(prophet_df["ds"][-1000:], prophet_df["y"][-1000:], label="True")
plt.plot(forecast["ds"][-1000:], forecast["yhat"][-1000:], label="Prophet Forecast")
plt.legend()
plt.title("Prophet vs True")
plt.show()

# --- Step 8: Attention Visualization (Simplified) ---
# For demo: visualize embedding weights as proxy
weights = model.embedding.weight.detach().numpy()
plt.imshow(weights, cmap="viridis", aspect="auto")
plt.colorbar()
plt.title("Attention Proxy: Embedding Weights")
plt.xlabel("hidden Units")
plt.ylabel("Input Features")
plt.show()

/tmp/ipython-input-1009972967.py:26: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["timestamp"] = pd.date_range(start="2020-01-01", periods=timesteps, freq="H")
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
